# Projection model

Trains models to project each player's 2025-26 fantasy points using only data from earlier seasons, and compares them with two simple baselines: last season's total (draft strategy 1) and a 3-season weighted average (strategy 2).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # make the project root importable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import config
from src.features import MODEL_FEATURES, TARGET, build_model_table, split_by_season
from src.model import evaluate, train_random_forest, train_xgboost, tune_random_forest, tune_xgboost

sns.set_theme(style="whitegrid")

In [2]:
model_table = build_model_table()
train, test = split_by_season(model_table)
print(f"Train: {len(train)} rows | Test: {len(test)} rows ({config.TEST_SEASON})")

Train: 1358 rows | Test: 347 rows (2025-26)


## Baselines

In [3]:
results = pd.DataFrame({
    "Last season (strategy 1)": evaluate(test[TARGET], test["FANTASY_PTS_LAG1"]),
    "3-season average (strategy 2)": evaluate(test[TARGET], test["FANTASY_PTS_AVG3"]),
}).T
results.round(0)

,MAE,RMSE
Last season (strategy 1),740.0,974.0
3-season average (strategy 2),740.0,956.0


## Tuning with time-based cross-validation

Settings are chosen using only the training seasons: train on earlier seasons, score on the next one, and average across three rounds. The 2025-26 test season is not used until the final comparison.

In [ ]:
rf_search = tune_random_forest(train)
xgb_search = tune_xgboost(train)


def cv_summary(search, name, defaults):
    """Cross-validated MAE for the original settings and for the best settings found."""
    scores = pd.DataFrame(search.cv_results_["params"])
    scores["CV MAE"] = -search.cv_results_["mean_test_score"]
    is_default = (scores[list(defaults)] == pd.Series(defaults)).all(axis=1)
    return {
        "Model": name,
        "CV MAE, original settings": scores.loc[is_default, "CV MAE"].iloc[0],
        "CV MAE, tuned": scores["CV MAE"].min(),
        "Best settings": search.best_params_,
    }


tuning = pd.DataFrame([
    cv_summary(rf_search, "Random Forest", {"min_samples_leaf": 5, "max_features": 1.0}),
    cv_summary(xgb_search, "XGBoost", {"max_depth": 4, "learning_rate": 0.05, "n_estimators": 300}),
]).set_index("Model")
pd.set_option("display.max_colwidth", None)
tuning.round(0)

,"CV MAE, original settings","CV MAE, tuned",Best settings
Model,,,
Random Forest,589.0,587.0,"{'max_features': 0.5, 'min_samples_leaf': 5}"
XGBoost,615.0,585.0,"{'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 300}"


Tuning barely changed the Random Forest (CV MAE 589 to 587) but improved XGBoost substantially (615 to 585) with shallower trees and a slower learning rate. The tuned models are effectively tied, so the Random Forest is used for projections because its individual trees also provide the floor and ceiling.

## Random Forest
Model settings come from config.py and were chosen by the cross-validation in the tuning section below.

In [4]:
rf = train_random_forest(train)
rf_test_pred = rf.predict(test[MODEL_FEATURES])
results.loc["Random Forest"] = pd.Series(evaluate(test[TARGET], rf_test_pred))

rf_train_mae = evaluate(train[TARGET], rf.predict(train[MODEL_FEATURES]))["MAE"]
print(f"Random Forest MAE on training rows: {rf_train_mae:,.0f}")
results.round(0)

Random Forest MAE on training rows: 393


,MAE,RMSE
Last season (strategy 1),740.0,974.0
3-season average (strategy 2),740.0,956.0
Random Forest,670.0,857.0


The Random Forest misses by 669 points on average, which is about 10% better than either baseline (740). Its RMSE is about 10% lower too (856 vs. 956), so it also makes fewer large misses.

## XGBoost and model comparison

In [5]:
xgb = train_xgboost(train)
xgb_test_pred = xgb.predict(test[MODEL_FEATURES])
results.loc["XGBoost"] = pd.Series(evaluate(test[TARGET], xgb_test_pred))

baseline_mae = results.loc["Last season (strategy 1)", "MAE"]
comparison = results.copy()
comparison["MAE better than last season"] = (1 - comparison["MAE"] / baseline_mae).map("{:.0%}".format)
comparison.round(0)

,MAE,RMSE,MAE better than last season
Last season (strategy 1),740.0,974.0,0%
3-season average (strategy 2),740.0,956.0,0%
Random Forest,670.0,857.0,9%
XGBoost,666.0,855.0,10%
